# Практическое занятие № 20. Полносвязная нейросеть в PyTorch
**ТюмГУ · «Технологии искусственного интеллекта — 1» · 90 минут**  
Автор учебных материалов: В. А. Мазур, доцент ШКН ТюмГУ.

**Результат:** собрать MLP, обучить её на прогнозе задержки доставки, использовать `autograd`, `DataLoader`, `BCEWithLogitsLoss` и Adam; контролировать обобщающую способность и сравнить с DummyClassifier и LogisticRegression на том же разбиении.

Если рядом есть `orders_dirty.csv`, будет выбран он; иначе — `demo_orders_19_20.csv` (отдельные синтетические данные). В Colab загрузите CSV во Files. У учебного набора не следует трактовать показатели как производственные результаты.

## 1. Настройка, проверка тензоров и autograd (0–18 мин)
Полносвязный нейрон: \(z=Wx+b\). `nn.BCEWithLogitsLoss()` объединяет sigmoid с логарифмической функцией потерь численно устойчивым способом; **не добавляйте `Sigmoid()` в конец модели**, если используете эту функцию. Студент должен прокомментировать, почему `loss.backward()` вычисляет градиенты по всей вычислительной схеме.

In [ ]:
from pathlib import Path
from copy import deepcopy
import numpy as np,pandas as pd,matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import TensorDataset,DataLoader
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler,OneHotEncoder
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score,f1_score,average_precision_score,confusion_matrix,log_loss
print('torch:',torch.__version__,'device:','cpu')
SEED=42
np.random.seed(SEED);torch.manual_seed(SEED)
try: torch.set_num_threads(2)
except RuntimeError: pass
device=torch.device('cpu')

In [ ]:
x=torch.tensor([[2.0,1.0]],requires_grad=True)
w=torch.tensor([[0.4],[-0.2]],requires_grad=True)
b=torch.tensor([-0.1],requires_grad=True)
z=x@w+b
z.sum().backward()
print('z:',z.item(),'gradient dz/dx:',x.grad,'dz/dw:',w.grad)
assert torch.allclose(w.grad,torch.tensor([[2.0],[1.0]]))

## 2. Аудит и разделение по времени (18–29 мин)
Единица наблюдения — заказ; метка \(y=late\) появляется после доставки. Для прогноза при оформлении допустимы только известные поля: расстояние, очередь, обещанный срок, загрузка курьера (если известна), погода, зона и час создания. `actual_minutes` используют только для проверки исторической метки. Разделение по времени: 60% train / 20% validation / 20% test. Проверки корректности **до** обучения.

In [ ]:
dataset=Path('orders_dirty.csv') if Path('orders_dirty.csv').exists() else Path('demo_orders_19_20.csv')
if not dataset.exists():
    raise FileNotFoundError('Загрузите orders_dirty.csv или demo_orders_19_20.csv')
df=pd.read_csv(dataset,encoding='utf-8-sig')
required={'order_id','created_at','distance_km','queue_size','zone','weather','promised_minutes','actual_minutes','late'}
assert required<=set(df.columns),f'Нет столбцов: {required-set(df.columns)}'
print('Источник:',dataset.name,'сырая таблица:',df.shape)
raw_n=len(df);df=df.drop_duplicates().copy()
for col in ['distance_km','queue_size','promised_minutes','actual_minutes','late']:
    df[col]=pd.to_numeric(df[col],errors='coerce')
if 'courier_load' not in df.columns:
    print('Нет courier_load: этот необязательный для урока признак пропускаем')
else:
    df['courier_load']=pd.to_numeric(df['courier_load'],errors='coerce')
df['created_at']=pd.to_datetime(df['created_at'],errors='coerce')
df=df.dropna(subset=['created_at','late']).copy()
df=df[df.late.isin([0,1])].copy()
can_check=df.actual_minutes.notna()&df.promised_minutes.notna()
conflict=can_check&(df.late != ((df.actual_minutes-df.promised_minutes)>15).astype(int))
print('Полных дубликатов:',raw_n-len(pd.read_csv(dataset,encoding='utf-8-sig').drop_duplicates()),'конфликтов метки:',int(conflict.sum()))
df=df.loc[~conflict].drop_duplicates(subset=['order_id'],keep='first').sort_values('created_at').reset_index(drop=True)
df['order_hour']=df['created_at'].dt.hour.astype(float)
n=len(df);ntrain=int(.60*n);nval=int(.20*n)
assert ntrain>20 and nval>6 and n-ntrain-nval>6
tr,va,te=df.iloc[:ntrain],df.iloc[ntrain:ntrain+nval],df.iloc[ntrain+nval:]
assert set(tr.order_id).isdisjoint(va.order_id) and set(tr.order_id).isdisjoint(te.order_id) and set(va.order_id).isdisjoint(te.order_id)
assert tr.created_at.max()<=va.created_at.min() and va.created_at.max()<=te.created_at.min()
num_cols=['distance_km','queue_size','promised_minutes','order_hour']
if 'courier_load' in df.columns: num_cols.append('courier_load')
cat_cols=['zone','weather']
print('N:',[len(tr),len(va),len(te)],'late ratio:',[round(v.late.mean(),3) for v in [tr,va,te]])

## 3. Безопасная предобработка (29–39 мин)
`ColumnTransformer`: для числовых полей медиана и стандартизация; для категорий мода и one-hot. **Обучаем `fit` только на train.** В PyTorch преобразуем результат в `torch.float32`. Списки признаков зафиксированы до просмотра test.

In [ ]:
try:
    enc=OneHotEncoder(handle_unknown='ignore',sparse_output=False)
except TypeError:
    enc=OneHotEncoder(handle_unknown='ignore',sparse=False)
prep=ColumnTransformer([
    ('num',Pipeline([('imputer',SimpleImputer(strategy='median')),('scaler',StandardScaler())]),num_cols),
    ('cat',Pipeline([('imputer',SimpleImputer(strategy='most_frequent')),('onehot',enc)]),cat_cols)
],remainder='drop')
Xtr=np.asarray(prep.fit_transform(tr[num_cols+cat_cols]),dtype=np.float32)
Xva=np.asarray(prep.transform(va[num_cols+cat_cols]),dtype=np.float32)
Xte=np.asarray(prep.transform(te[num_cols+cat_cols]),dtype=np.float32)
ytr=tr.late.to_numpy(dtype=np.float32).reshape(-1,1)
yva=va.late.to_numpy(dtype=np.float32).reshape(-1,1)
yte=te.late.to_numpy(dtype=np.float32).reshape(-1,1)
assert np.isfinite(Xtr).all() and np.isfinite(Xva).all() and np.isfinite(Xte).all()
print('X shape train/val/test:',Xtr.shape,Xva.shape,Xte.shape)
train_loader=DataLoader(TensorDataset(torch.from_numpy(Xtr),torch.from_numpy(ytr)),batch_size=32,shuffle=True,generator=torch.Generator().manual_seed(SEED))
valid_loader=DataLoader(TensorDataset(torch.from_numpy(Xva),torch.from_numpy(yva)),batch_size=128,shuffle=False)

## 4. Модель, функция потерь и оптимизатор (39–51 мин)
MLP: \(d_{in}\rightarrow 32\rightarrow 16\rightarrow 1\) с ReLU. Выход — **логит**, не вероятность. Параметры обучаются методом `Adam`. Количество параметров для Linear: \(d_{in}\cdot d_{out}+d_{out}\).

In [ ]:
class DeliveryMLP(nn.Module):
    def __init__(self,n_inputs):
        super().__init__()
        self.network=nn.Sequential(nn.Linear(n_inputs,32),nn.ReLU(),
                                   nn.Linear(32,16),nn.ReLU(),nn.Linear(16,1))
    def forward(self,x):return self.network(x)
model=DeliveryMLP(Xtr.shape[1]).to(device)
criterion=nn.BCEWithLogitsLoss()
optimizer=torch.optim.Adam(model.parameters(),lr=1e-3)
print(model);print('Обучаемых параметров:',sum(p.numel() for p in model.parameters()))

## 5. Mini-batch обучение, валидация и ранняя остановка (51–69 мин)
Цикл по пакетам: `zero_grad → forward → loss → backward → step`. Для validation: `model.eval()` и `torch.no_grad()`. Лучшее состояние сохраняется **по validation loss**, test в этом блоке недоступен. Для ускорения практикума всё работает на CPU.

In [ ]:
def mean_loss(loader):
    model.eval(); total=0; count=0
    with torch.no_grad():
        for xb,yb in loader:
            l=criterion(model(xb.to(device)),yb.to(device))
            total+=float(l)*len(xb);count+=len(xb)
    return total/count
history=[];best=float('inf');best_state=None;bad=0
PATIENCE=12
for epoch in range(80):
    model.train();total=0.;count=0
    for xb,yb in train_loader:
        optimizer.zero_grad(set_to_none=True)
        logits=model(xb.to(device))
        loss=criterion(logits,yb.to(device))
        loss.backward()
        optimizer.step()
        total+=float(loss.detach())*len(xb);count+=len(xb)
    vl=mean_loss(valid_loader)
    tl=total/count
    history.append((epoch+1,tl,vl))
    if vl < best-1e-4:
        best=vl;best_state=deepcopy(model.state_dict());bad=0
    else:
        bad+=1
    if bad>=PATIENCE:break
assert best_state is not None
model.load_state_dict(best_state)
hist=pd.DataFrame(history,columns=['epoch','train_bce','validation_bce'])
print('Итераций:',len(hist),'лучший validation BCE:',round(best,4))
ax=hist.plot(x='epoch',y=['train_bce','validation_bce'],figsize=(8,3.5),title='MLP: кривые обучения')
ax.set(ylabel='BCE');ax.grid(alpha=.3);plt.show()

## 6. Сравнение с baseline на validation, финальная проверка test (69–82 мин)
Сравнивать модели можно только на **одном разбиении и одних входных данных**. Сначала сравните на validation. В рамках упражнения архитектура и порог 0,5 фиксированы до test; никаких новых настроек после тестирования.

In [ ]:
def nn_probability(x):
    model.eval()
    with torch.no_grad():
        return torch.sigmoid(model(torch.from_numpy(x))).numpy().ravel()

def score_row(title,y,p,thr=.5):
    pred=(p>=thr).astype(int)
    return {'model':title,'N':len(y),'accuracy':accuracy_score(y,pred),
            'F1':f1_score(y,pred,zero_division=0),
            'AP':average_precision_score(y,p) if len(np.unique(y))==2 else np.nan,
            'BCE':log_loss(y,np.clip(p,1e-7,1-1e-7),labels=[0,1])}
prior=DummyClassifier(strategy='prior').fit(Xtr,ytr.ravel())
linear=LogisticRegression(max_iter=1000,random_state=42).fit(Xtr,ytr.ravel())
models=[('Dummy',prior),('LogReg',linear)]
rows=[]
for name,clf in models:
    rows.append(score_row(name,yva.ravel(),clf.predict_proba(Xva)[:,list(clf.classes_).index(1)]))
rows.append(score_row('MLP (PyTorch)',yva.ravel(),nn_probability(Xva)))
valid_scores=pd.DataFrame(rows).sort_values('F1',ascending=False)
print('Validation — выбираем, не глядя на test')
display(valid_scores.round(3))
# Фиксируем модель/порог по установленному ДО теста правилу: сравнение обученной MLP с baseline;
# не переобучаем и не меняем архитектуру после следующего блока.
print('Порог фиксирован:',0.5,'; архитектура MLP и метод ранней остановки фиксированы.')

In [ ]:
test_rows=[]
for name,clf in models:
    test_rows.append(score_row(name,yte.ravel(),clf.predict_proba(Xte)[:,list(clf.classes_).index(1)]))
pte=nn_probability(Xte)
test_rows.append(score_row('MLP (PyTorch)',yte.ravel(),pte))
test_scores=pd.DataFrame(test_rows)
print('TEST: итоговая оценка без дальнейшего подбора параметров')
display(test_scores.round(3))
cm=confusion_matrix(yte.ravel(),(pte>=0.5).astype(int),labels=[0,1])
fig,ax=plt.subplots(figsize=(4,3.5));im=ax.imshow(cm,cmap='Blues')
ax.set_xticks([0,1]);ax.set_yticks([0,1]);ax.set(xlabel='Прогноз',ylabel='Факт',title='MLP: матрица ошибок (test)')
for (i,j),v in np.ndenumerate(cm):ax.text(j,i,str(v),ha='center',va='center',fontsize=13)
plt.show()
assert cm.sum()==len(yte)

## 7. Сохранение результатов и самостоятельные задания (82–90 мин)
Сохраните веса MLP и параметры подготовки, чтобы эксперимент можно было повторить. Сведения о версиях и входных полях — обязательная часть отчёта.

In [ ]:
import joblib
from pathlib import Path
folder=Path('results_20');folder.mkdir(exist_ok=True)
torch.save(model.state_dict(),folder/'mlp_state_dict.pt')
joblib.dump(prep,folder/'preprocessor.joblib')
hist.to_csv(folder/'history.csv',index=False)
valid_scores.to_csv(folder/'validation_metrics.csv',index=False)
test_scores.to_csv(folder/'test_metrics.csv',index=False)
print('Сохранены артефакты:',[p.name for p in folder.iterdir()])

**Самостоятельно:** (1) объясните размерность `Xtr` после one-hot; (2) сравните минимум две архитектуры **только по validation** — эксперимент организуйте на отдельных запусках до финального test; (3) объясните значение `model.train()`, `model.eval()`, `zero_grad()` и `no_grad()`; (4) сравните качество нейросети с LogisticRegression и Dummy, указав ограничения малого синтетического набора; (5) приведите две кривые loss и confusion matrix.

**Сдать:** `20_student.ipynb`, `Группа_Фамилия_З20_Отчет.docx` (2–4 страницы), папку `results_20` по требованию преподавателя.

**Оценивание (10 баллов):** корректная предобработка и разбиение — 2; архитектура и автодифференцирование — 2; цикл обучения/валидация — 3; сравнение и матрица ошибок — 2; воспроизводимость/выводы — 1.

**Контрольные вопросы:** Почему `BCEWithLogitsLoss` не требует `Sigmoid()` внутри модели? Чем validation отличается от test? Что такое mini-batch? Почему исходный ID нельзя подавать в модель? Когда имеет смысл выбрать логистическую регрессию вместо нейросети?